# Inter-peak cross-$g_2$

Temporal correlation of small-square ROI intensities at 320–350 K (scans 343, 344, 349, 350).

Each peak gets a ~15×15 square centered on the XPCS 2D-Gaussian fit. Auto-$g_2$ of each trace, cross-$g_2$ of every pair, and the (120)×(101) two-time map. A time-shuffle of one trace is the control (cross-$g_2$ should sit at 1).

This is **not** a spatial 2D image cross-correlation of the two peak crops.


In [ ]:
import sys
sys.path.append('../src/')

from pathlib import Path
import json
from itertools import combinations

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import Normalize

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    pass

from autocorrelations import g2_traces, twotime_traces
from postprocess import (
    FOUR_PEAK_SCANS,
    REFLECTION_ORDER,
    temperature_cmap,
    temperature_color,
    temperature_norm,
)


In [ ]:
def apply_paper_style():
    helvetica = font_manager.FontProperties(family='Helvetica')
    font_manager.findfont(helvetica, fallback_to_default=False)
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Helvetica', 'Arial'],
        'font.size': 9,
        'axes.labelsize': 11,
        'axes.titlesize': 11,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 7,
        'legend.frameon': False,
        'axes.linewidth': 1.0,
        'lines.linewidth': 1.2,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
        'xtick.major.size': 4.5,
        'ytick.major.size': 4.5,
        'xtick.top': True,
        'ytick.right': True,
        'mathtext.fontset': 'custom',
        'mathtext.rm': 'Helvetica',
        'mathtext.it': 'Helvetica:italic',
        'mathtext.bf': 'Helvetica:bold',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': 600,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.03,
    })


apply_paper_style()

cmap = temperature_cmap()
norm = temperature_norm()

HKL_COLORS = {
    '011': '#2E5E8C',
    '10-1': '#2F8A6B',
    '120': '#C97A2B',
    '101': '#9E3B4E',
}
HKL_LABELS = {
    '011': '(011)',
    '10-1': r'(10$\overline{1}$)',
    '120': '(120)',
    '101': '(101)',
}
FEATURE_PAIR = ('120', '101')
ROI_HALF = 7
HOT_PIXEL = 50.0
DPL = 6
TWOTIME_BINS = 300

config_path = Path('../configs/config_B10.json')
config = json.loads(config_path.read_text()) if config_path.exists() else {}
APS_BASE = Path(config['Base']) if config.get('Base') else Path('/Users/eriklamb/data/APS/8_ID_E/Na2B10H10')


In [ ]:
def _scalar(value):
    return float(np.asarray(value).reshape(-1)[0])


def find_result(T, scan, hkl):
    folder = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001' / 'results'
    hits = list(folder.glob(f'*({hkl})*.h5'))
    if not hits:
        raise FileNotFoundError(f'no ({hkl}) result for T={T} scan={scan}')
    return hits[0]


def find_raw(T, scan):
    folder = APS_BASE / f'A{scan}_NaBH_att000020_{T}K_001'
    hits = sorted(folder.glob('*_001.h5'))
    if not hits:
        raise FileNotFoundError(f'no raw detector file for T={T} scan={scan}')
    return hits[0]


def square_roi(row, col, half, n_row, n_col):
    r0 = max(0, int(round(row)) - half)
    r1 = min(n_row, int(round(row)) + half + 1)
    c0 = max(0, int(round(col)) - half)
    c1 = min(n_col, int(round(col)) + half + 1)
    return r0, r1, c0, c1


def load_trace(T, scan, hkl, half=ROI_HALF):
    result_path = find_result(T, scan, hkl)
    with h5py.File(result_path, 'r') as f:
        popt = np.asarray(f['peaks/peak_0/popt'][()], dtype=float).ravel()
        dt = _scalar(f['experimental_parameters/frame_spacing'][()])
    col, row = float(popt[1]), float(popt[2])
    raw = find_raw(T, scan)
    with h5py.File(raw, 'r') as f:
        nfr, n_row, n_col = f['entry/data/data'].shape
        r0, r1, c0, c1 = square_roi(row, col, half, n_row, n_col)
        crop = np.asarray(f['entry/data/data'][:, r0:r1, c0:c1], dtype=np.float64)
    crop[crop > HOT_PIXEL] = 0.0
    intensity = crop.mean(axis=(1, 2))
    return {
        'temperature': float(T),
        'scan': scan,
        'hkl': hkl,
        'intensity': intensity,
        'dt': dt,
        'row': row,
        'col': col,
        'roi': (r0, r1, c0, c1),
    }


def bin_twotime(C, nbin=TWOTIME_BINS):
    n = C.shape[0]
    if n <= nbin:
        return C
    step = n // nbin
    cut = step * nbin
    block = C[:cut, :cut].reshape(nbin, step, nbin, step)
    return block.mean(axis=(1, 3))


In [ ]:
records = {}
for T, scan in FOUR_PEAK_SCANS:
    records[T] = {}
    for hkl in REFLECTION_ORDER:
        rec = load_trace(T, scan, hkl)
        records[T][hkl] = rec
        r0, r1, c0, c1 = rec['roi']
        print(
            f"{T:g} K  ({hkl:5s})  n={rec['intensity'].size}  "
            f"roi={r1-r0}x{c1-c0}  <I>={rec['intensity'].mean():.3f}  dt={rec['dt']:.3g} s"
        )


In [ ]:
rng = np.random.default_rng(0)
g2_results = {}
twotime_feature = {}
contrast_rows = []

for T, scan in FOUR_PEAK_SCANS:
    g2_results[T] = {}
    dt = next(iter(records[T].values()))['dt']
    peaks = list(REFLECTION_ORDER)
    pairs = [(a, a) for a in peaks] + list(combinations(peaks, 2))
    for a, b in pairs:
        Ia = records[T][a]['intensity']
        Ib = records[T][b]['intensity']
        delay, g2, g2e = g2_traces(Ia, None if a == b else Ib, dt=dt, dpl=DPL)
        g2_results[T][(a, b)] = (delay, g2, g2e)
        if a != b:
            Ib_sh = rng.permutation(Ib)
            delay_s, g2_s, g2e_s = g2_traces(Ia, Ib_sh, dt=dt, dpl=DPL)
            g2_results[T][(a, b, 'shuffle')] = (delay_s, g2_s, g2e_s)
            contrast_rows.append((T, a, b, float(g2[0]), float(g2e[0]), float(g2_s[0])))
        else:
            contrast_rows.append((T, a, b, float(g2[0]), float(g2e[0]), np.nan))
    Ia = records[T][FEATURE_PAIR[0]]['intensity']
    Ib = records[T][FEATURE_PAIR[1]]['intensity']
    twotime_feature[T] = {
        'C': twotime_traces(Ia, Ib),
        'C_shuffle': twotime_traces(Ia, rng.permutation(Ib)),
        'dt': dt,
    }
    print(f'{T:g} K  (120)x(101)  g2(0)={g2_results[T][FEATURE_PAIR][1][0]:.4f}')


In [ ]:
temps = [T for T, _ in FOUR_PEAK_SCANS]
fig, axes = plt.subplots(3, 4, figsize=(11.2, 8.4))

for col, T in enumerate(temps):
    dt = next(iter(records[T].values()))['dt']
    ax_I = axes[0, col]
    ax_g = axes[1, col]
    ax_c = axes[2, col]
    n = next(iter(records[T].values()))['intensity'].size
    t = np.arange(n) * dt

    for hkl in REFLECTION_ORDER:
        I = records[T][hkl]['intensity']
        Iplot = I / np.median(I)
        ax_I.plot(t, Iplot, color=HKL_COLORS[hkl], lw=0.7, label=HKL_LABELS[hkl])
    ax_I.set_title(f'{T:g} K', color=temperature_color(T, cmap=cmap, norm=norm))
    ax_I.set_xlabel('t (s)' if col == 0 else '')
    if col == 0:
        ax_I.set_ylabel(r'$I(t)/\mathrm{median}$')
        ax_I.legend(loc='upper right', handlelength=1.1)

    for hkl in REFLECTION_ORDER:
        delay, g2, g2e = g2_results[T][(hkl, hkl)]
        ax_g.errorbar(
            delay[1:], g2[1:], yerr=g2e[1:], fmt='-', color=HKL_COLORS[hkl],
            lw=1.0, elinewidth=0.6, capsize=0, label=HKL_LABELS[hkl],
        )
    delay, g2, g2e = g2_results[T][FEATURE_PAIR]
    ax_g.errorbar(
        delay, g2, yerr=g2e, fmt='k-', lw=1.4, elinewidth=0.6, capsize=0,
        label=r'(120)$\times$(101)',
    )
    delay_s, g2_s, _ = g2_results[T][FEATURE_PAIR + ('shuffle',)]
    ax_g.plot(delay_s, g2_s, color='0.55', ls='--', lw=1.0, label='shuffle')
    ax_g.axhline(1.0, color='0.75', lw=0.7, zorder=0)
    ax_g.set_xscale('log')
    ax_g.set_xlabel(r'$\tau$ (s)' if col == 0 else '')
    if col == 0:
        ax_g.set_ylabel(r'$g_2(\tau)$')
        ax_g.legend(loc='upper right', handlelength=1.2)

    C = bin_twotime(twotime_feature[T]['C'])
    tmax = n * dt
    im = ax_c.imshow(
        C, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1,
        extent=[0, tmax, 0, tmax], interpolation='nearest',
    )
    ax_c.set_xlabel(r'$t_2$ (s)')
    if col == 0:
        ax_c.set_ylabel(r'$t_1$ (s)')
    ax_c.set_title(r'(120)$\times$(101)', fontsize=9)
    if col == 3:
        cbar = fig.colorbar(im, ax=ax_c, fraction=0.046, pad=0.04)
        cbar.set_label(r'$C_{ab}(t_1,t_2)$')

fig.tight_layout(w_pad=0.8, h_pad=1.0)
out = Path('../figures')
out.mkdir(exist_ok=True)
fig.savefig(out / 'peak_cross_g2.pdf')
fig.savefig(out / 'peak_cross_g2.png')
print('saved', out / 'peak_cross_g2.pdf')


In [ ]:
print(f"{'T (K)':>6}  {'a':>6}  {'b':>6}  {'g2(0)':>8}  {'±':>8}  {'shuffle':>8}")
print('-' * 52)
lines = ['T_K,hkl_a,hkl_b,g2_0,g2_0_err,g2_0_shuffle']
for T, a, b, g0, ge, gs in contrast_rows:
    sh = '' if not np.isfinite(gs) else f'{gs:8.4f}'
    print(f'{T:6.0f}  {a:>6}  {b:>6}  {g0:8.4f}  {ge:8.4f}  {sh:>8}')
    gs_txt = '' if not np.isfinite(gs) else f'{gs:.6f}'
    lines.append(f'{T:.0f},{a},{b},{g0:.6f},{ge:.6f},{gs_txt}')
csv_path = Path('../figures/peak_cross_g2_contrast.csv')
csv_path.write_text('\n'.join(lines) + '\n')
print('saved', csv_path)
